# Predicting Stellar Class — Preprocessing & EDA

Kaggle Playground Series S6E6. Multiclass classification: `GALAXY` / `STAR` / `QSO`.
Metric — **balanced accuracy** (mean per-class recall), so class imbalance matters.

Notebook goals:
1. Load and validate the data (shapes, dtypes, missing values, duplicates, train/test consistency).
2. Full EDA: target variable, numeric and categorical features, relation to the class, correlations, anomalies.
3. Build a clean feature set for modeling and persist it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

PURPLE_BLUE = ['#3B2F8F', '#6A5ACD', '#9D7BE0', '#4169E1', '#7B68EE', '#5B8DEF']
CLASS_PALETTE = {'GALAXY': '#3B2F8F', 'QSO': '#6A5ACD', 'STAR': '#9D7BE0'}
SEQ_CMAP = 'BuPu'

sns.set_theme(style='whitegrid', palette=PURPLE_BLUE)
plt.rcParams['figure.dpi'] = 110

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    train_paths = sorted(Path('/kaggle/input').rglob('train.csv'))
    DATA_DIR = train_paths[0].parent
    OUT_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../docs/dataset')
    OUT_DIR = Path('../data_processed')
print('Environment:', 'Kaggle' if ON_KAGGLE else 'local')
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR :', OUT_DIR)

TARGET = 'class'
ID = 'id'
CLASSES = ['GALAXY', 'QSO', 'STAR']


def reduce_mem(df):
    '''
    Downcast numeric columns (float64 -> float32, int64 -> smallest int)
    to roughly halve memory usage, which matters under Kaggle RAM limits.
    '''
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

## 1. Data loading

In [ ]:
train = reduce_mem(pd.read_csv(DATA_DIR / 'train.csv'))
test = reduce_mem(pd.read_csv(DATA_DIR / 'test.csv'))
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train:', train.shape)
print('test :', test.shape)
print('sub  :', sample_sub.shape)
print('train memory: %.1f MB' % (train.memory_usage(deep=True).sum() / 1024**2))
train.head()

**Observations.** Train has ~577k rows and 12 columns; test has ~247k rows and 11 columns (no `class`). The feature schema matches between train and test, and `sample_submission` aligns with the test size, so the split is well formed.

## 2. Basic data quality checks

In [ ]:
train.info(show_counts=True)

**Observations.** 8 numeric columns (`float64`), 1 integer `id`, and 3 `object` columns (`spectral_type`, `galaxy_population`, `class`). Every column is fully populated (577,347 non-null), confirming no missing data in train at the dtype level.

In [ ]:
miss = pd.DataFrame({
    'train_nulls': train.isnull().sum(),
    'test_nulls': test.reindex(columns=train.columns).isnull().sum(),
})
print(miss)
print('\nTotal nulls train:', train.isnull().sum().sum(), '| test:', test.isnull().sum().sum())

**Observations.** No missing values anywhere except `class` in test, which is expected since the target is what we predict. No imputation strategy is required.

In [ ]:
print('Duplicate ids in train:', train[ID].duplicated().sum())
print('Duplicate ids in test :', test[ID].duplicated().sum())
print('Shared ids train/test:', len(set(train[ID]) & set(test[ID])))

train_feats = set(train.columns) - {TARGET}
test_feats = set(test.columns)
print('\nColumns only in train (excluding class):', train_feats - test_feats)
print('Columns only in test :', test_feats - train_feats)

feat_cols = [c for c in train.columns if c not in (ID, TARGET)]
print('\nFully duplicated feature rows (train):', train.duplicated(subset=feat_cols).sum())

**Observations.** No duplicate `id` in either split, no overlap of ids between train and test, and no fully duplicated feature rows. The feature columns are identical across splits (only `class` is train-exclusive), so the data is consistent and leak-free.

In [ ]:
num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
cat_cols = ['spectral_type', 'galaxy_population']
print('Numeric    :', num_cols)
print('Categorical:', cat_cols)
train[num_cols].describe().T

**Observations.** The five photometric magnitudes (`u, g, r, i, z`) sit in a similar 12–28 range, hinting at strong correlation among them. `redshift` is heavily right-skewed (mean 0.72, max 7.0) with a small negative minimum. `alpha` spans the full 0–360° RA range and `delta` the declination range, as expected for sky coordinates.

## 3. Target variable

In [ ]:
vc = train[TARGET].value_counts()
vc_norm = train[TARGET].value_counts(normalize=True)
print(pd.DataFrame({'count': vc, 'share': vc_norm.round(4)}))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=vc.index, y=vc.values, ax=ax[0],
            hue=vc.index, legend=False,
            palette=[CLASS_PALETTE.get(c, '#6A5ACD') for c in vc.index])
ax[0].set_title('Count by class')
ax[1].pie(vc.values, labels=vc.index, autopct='%1.1f%%', startangle=90,
          colors=[CLASS_PALETTE.get(c, '#6A5ACD') for c in vc.index])
ax[1].set_title('Class shares')
plt.tight_layout(); plt.show()

**Observations.** The target is clearly imbalanced: GALAXY dominates (~65%), QSO is mid-sized (~20%), and STAR is the minority (~14%). Because the metric is balanced accuracy, the minority STAR class must be protected — use `StratifiedKFold` and `balanced_accuracy_score`, and consider class weighting.

## 4. Numeric features: distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(train[col], bins=80, ax=ax, color='#6A5ACD')
    ax.set_title(col)
plt.suptitle('Numeric feature distributions (train)', y=1.02)
plt.tight_layout(); plt.show()

**Observations.** The magnitudes are roughly bell-shaped with mild left skew. `redshift` is strongly right-skewed with a spike near zero (stars) and a long tail (quasars) — this multimodality already foreshadows its discriminative power. `alpha`/`delta` are broad and fairly flat, typical of survey footprint coverage.

In [ ]:
for col in num_cols:
    neg = (train[col] < 0).sum()
    print(f'{col:9s}  min={train[col].min():10.4f}  max={train[col].max():10.4f}  <0: {neg}')

**Observations.** Negative `delta` values (~103k) are just southern-hemisphere declinations and are valid. Negative `redshift` (~9k) is physically plausible (peculiar velocities / measurement noise) and should be kept. Only a single negative `u` magnitude appears — a likely outlier, but not worth special handling given its rarity.

## 5. Numeric features by class

In [ ]:
print(train.groupby(TARGET)['redshift'].describe().round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for cls in CLASSES:
    sns.kdeplot(train.loc[train[TARGET] == cls, 'redshift'].clip(-0.5, 4),
                label=cls, ax=ax[0], fill=True, alpha=0.3, color=CLASS_PALETTE[cls])
ax[0].set_title('redshift by class (clip -0.5..4)'); ax[0].legend()
sns.boxplot(data=train, x=TARGET, y='redshift', order=CLASSES, ax=ax[1],
            hue=TARGET, legend=False,
            palette=[CLASS_PALETTE[c] for c in CLASSES])
ax[1].set_ylim(-0.5, 4); ax[1].set_title('redshift boxplot by class')
plt.tight_layout(); plt.show()

**Observations.** `redshift` separates the classes almost perfectly: STAR clusters at ≈ 0, GALAXY occupies a moderate range, and QSO sits at high redshift. This is the single most powerful predictor and should drive most of the model's accuracy.

In [ ]:
other_num = [c for c in num_cols if c != 'redshift']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), other_num):
    sns.boxplot(data=train, x=TARGET, y=col, order=CLASSES, ax=ax, showfliers=False,
                hue=TARGET, legend=False,
                palette=[CLASS_PALETTE[c] for c in CLASSES])
    ax.set_title(col)
for ax in axes.ravel()[len(other_num):]:
    ax.axis('off')
plt.suptitle('Numeric features by class (plot outliers hidden)', y=1.02)
plt.tight_layout(); plt.show()

**Observations.** The raw magnitudes overlap heavily across classes — individually they are weak separators. The class medians differ only modestly, which reinforces that derived colors (band differences) should be more informative than the magnitudes themselves.

## 6. Categorical features

In [ ]:
for col in cat_cols:
    print(f'=== {col} ===')
    print(train[col].value_counts())
    print('unique in test but missing from train:', set(test[col].unique()) - set(train[col].unique()))
    print()

**Observations.** `spectral_type` has 4 levels (M most common, O/B rarest) and `galaxy_population` is binary. No unseen categories appear in test, so a simple ordinal/category encoding is safe with no fallback needed for unknown values.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, col in zip(ax, cat_cols):
    ct = pd.crosstab(train[col], train[TARGET], normalize='index')[CLASSES]
    ct.plot(kind='bar', stacked=True, ax=a, color=[CLASS_PALETTE[c] for c in CLASSES])
    a.set_title(f'Class share within {col}'); a.set_ylabel('share')
    a.legend(title='class', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout(); plt.show()

print(pd.crosstab(train['spectral_type'], train[TARGET], normalize='index').round(3))

**Observations.** Class composition shifts strongly with `spectral_type`: the O/B (hot) type is dominated by QSO, while cooler types lean toward GALAXY/STAR. `galaxy_population` also splits the classes meaningfully. Both categoricals carry real signal and should be kept as features.

## 7. Correlations and spatial distribution

In [ ]:
corr = train[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap=SEQ_CMAP, square=True)
plt.title('Numeric feature correlations')
plt.tight_layout(); plt.show()

**Observations.** The `u, g, r, i, z` bands are very highly correlated with one another (multicollinearity), confirming the earlier hypothesis. Color differences (`u-g`, `g-r`, …) decorrelate this block and are expected to be more informative than the raw magnitudes.

In [ ]:
samp = train.sample(40000, random_state=42)
plt.figure(figsize=(11, 5))
sns.scatterplot(data=samp, x='alpha', y='delta', hue=TARGET, hue_order=CLASSES,
                palette=CLASS_PALETTE, s=6, alpha=0.4, linewidth=0)
plt.title('Object positions on the sky (alpha vs delta), 40k sample')
plt.tight_layout(); plt.show()

**Observations.** Classes are well mixed across the sky with no obvious spatial clustering by class. The `alpha`/`delta` coordinates therefore carry little direct class signal and are unlikely to be strong predictors on their own.

## 8. train vs test comparison (drift)
The dataset is synthetic — let's confirm that feature distributions are consistent between train and test.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.kdeplot(train[col].clip(train[col].quantile(.001), train[col].quantile(.999)),
                label='train', ax=ax, color='#3B2F8F')
    sns.kdeplot(test[col].clip(test[col].quantile(.001), test[col].quantile(.999)),
                label='test', ax=ax, color='#9D7BE0')
    ax.set_title(col); ax.legend()
plt.suptitle('train vs test: feature distributions', y=1.02)
plt.tight_layout(); plt.show()

**Observations.** The train and test distributions overlap almost perfectly for every numeric feature — no meaningful covariate shift. A model validated on train CV should generalize reliably to the test set without drift-correction tricks.

## 9. Preprocessing and feature engineering
- `id` is not used as a feature.
- Build color indices from the photometry (physically meaningful SDSS features).
- Encode categorical features (ordered codes; for gradient boosting they can be passed as `category`).
- Encode the target into integer labels.

In [ ]:
def add_features(df):
    '''
    Add SDSS color indices: differences between adjacent photometric bands.
    These colors are the standard, physically meaningful features for SDSS
    object classification and carry more signal than raw magnitudes.
    '''
    df = df.copy()
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_z'] = df['u'] - df['z']
    return df

train_fe = add_features(train)
test_fe = add_features(test)
color_cols = ['u_g', 'g_r', 'r_i', 'i_z', 'u_z']

fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, col in zip(axes, color_cols):
    sns.boxplot(data=train_fe, x=TARGET, y=col, order=CLASSES, ax=ax, showfliers=False,
                hue=TARGET, legend=False,
                palette=[CLASS_PALETTE[c] for c in CLASSES])
    ax.set_title(col)
plt.tight_layout(); plt.show()

**Observations.** The color indices show clearer class separation than the raw bands — `u_g` and `u_z` in particular shift noticeably between classes. This validates adding the colors as engineered features.

In [ ]:
spectral_order = {'O/B': 0, 'A/F': 1, 'G/K': 2, 'M': 3}
pop_map = {'Blue_Cloud': 0, 'Red_Sequence': 1}

for df in (train_fe, test_fe):
    df['spectral_type_code'] = df['spectral_type'].map(spectral_order)
    df['galaxy_population_code'] = df['galaxy_population'].map(pop_map)

class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}
train_fe['target'] = train_fe[TARGET].map(class_to_int)

feature_cols = num_cols + color_cols + ['spectral_type_code', 'galaxy_population_code']
print('Final features (%d):' % len(feature_cols))
print(feature_cols)
print('\nNull check after encoding:',
      train_fe[feature_cols].isnull().sum().sum(), test_fe[feature_cols].isnull().sum().sum())

**Observations.** 15 final features (8 numeric + 5 colors + 2 encoded categoricals) with zero nulls in both train and test after encoding. The ordinal `spectral_type` mapping (hot → cool) preserves a physically meaningful ordering. The feature matrix is model-ready.

## 10. Quick baseline to check the signal

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

X = train_fe[feature_cols]
y = train_fe['target']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42, class_weight='balanced')

oof = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)
print('OOF balanced accuracy:', round(balanced_accuracy_score(y, oof), 5))
print('\n', classification_report(y, oof, target_names=CLASSES))

**Observations.** The OOF balanced accuracy of ~0.929 confirms a strong signal even from a plain ExtraTrees baseline. As expected, GALAXY and QSO have high recall (~0.96–0.97) while STAR, the minority class, lags (~0.85) — STAR recall is the main lever for improving the balanced-accuracy score.

In [ ]:
cm = confusion_matrix(y, oof, normalize='true')
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='.3f', cmap=SEQ_CMAP, xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (row-wise recall)')
plt.tight_layout(); plt.show()

model.fit(X, y)
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
plt.figure(figsize=(7, 5)); imp.plot(kind='barh', color='#6A5ACD'); plt.title('Feature importance (ExtraTrees)')
plt.tight_layout(); plt.show()

**Observations.** The confusion matrix shows STAR is the hardest class, with its misclassifications leaking mainly into the dominant classes. Feature importance is led by `redshift` and `spectral_type_code`, with the color indices contributing moderately and the sky coordinates near the bottom — consistent with every earlier EDA finding.

## 11. Saving outputs and submission

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_fe[[ID] + feature_cols + ['target', TARGET]].to_parquet(OUT_DIR / 'train_fe.parquet', index=False)
test_fe[[ID] + feature_cols].to_parquet(OUT_DIR / 'test_fe.parquet', index=False)

test_pred = model.predict(test_fe[feature_cols])
submission = pd.DataFrame({ID: test_fe[ID], TARGET: pd.Series(test_pred).map(int_to_class)})
submission.to_csv(OUT_DIR / 'submission.csv', index=False)

print('Saved to', OUT_DIR.resolve())
print('train_fe:', train_fe[[ID] + feature_cols + ['target']].shape, '| test_fe:', test_fe[[ID] + feature_cols].shape)
print('submission:', submission.shape)
submission.head()

**Observations.** Outputs land in `OUT_DIR` (`/kaggle/working` on Kaggle, local folder otherwise): the engineered `train_fe`/`test_fe` Parquet files plus a competition-ready `submission.csv` (`id`, `class`) from the baseline model. Downstream modeling notebooks can load the Parquet directly without repeating preprocessing.

## EDA conclusions
- The data is clean: no missing values, no duplicate `id`, train/test columns are consistent.
- Classes are imbalanced (GALAXY ~65%, QSO ~20%, STAR ~14%) → strictly use `balanced_accuracy` + `StratifiedKFold`.
- `redshift` is the strongest feature (STAR ≈ 0, GALAXY moderate, QSO high); `spectral_type` is also highly informative.
- The `u, g, r, i, z` bands are strongly correlated — the added color differences provide extra signal.
- train/test distributions are consistent — no serious drift is visible.
- `alpha/delta` coordinates carry only a weak signal, and `id` is excluded from the features.